- Delta lake and spark is less involved than iceberg
- you dont have to put in the maven package coordinates
- you do need though the python package delta-spark
- but other than tha, the config is much easier than iceberg

In [ ]:
import shutil
shutil.rmtree("./delta_warehouse", ignore_errors=True)

In [ ]:
import os
import pyspark
from delta import *

delta_warehouse = os.path.abspath("./delta_warehouse")

builder = (
    pyspark.sql.SparkSession.builder.appName("MyApp") 
        .config("spark.driver.host", "127.0.0.1") 
        .config("spark.driver.bindAddress", "127.0.0.1") 
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") 
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") 
        .config("spark.sql.warehouse.dir", delta_warehouse) 
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()

In [ ]:
schema_nm = "purch_ord"
spark.sql(f"create schema if not exists {schema_nm}")

In [ ]:
sql = f"""
CREATE OR REPLACE TABLE {schema_nm}.orders (
    order_id BIGINT,
    order_date DATE,
    customer_id BIGINT,
    total_amount DOUBLE
) USING DELTA
"""
spark.sql(sql)
spark.sql(f"insert into {schema_nm}.orders values (1, current_date(), 1001, 250.75)")

In [ ]:
spark.sql(f"select * from {schema_nm}.orders").show()

In [ ]:
# export to csv
spark.sql(f"select * from {schema_nm}.orders").write.mode("overwrite").option("header", "true").csv("./orders_csv")